# 02_functions_oop — Функции и ООП для ML Engineer Interview

Этот ноутбук — структурированный конспект и практикум для подготовки к собеседованию ML Engineer.

**Формат каждого раздела:**
1. Теория (глубоко, без воды)
2. Кодовые примеры
3. Разбор кода
4. Производительность и память
5. Interview-вопросы
6. Мини-задачи
7. Edge cases


## 1) Аргументы функций: positional, keyword, default, `*args`, `**kwargs`

### Теория
В Python сигнатура функции определяет контракт вызова:
- **Positional arguments**: передаются по позиции.
- **Keyword arguments**: передаются по имени параметра.
- **Default arguments**: позволяют делать параметры опциональными.
- `*args`: собирает лишние позиционные аргументы в `tuple`.
- `**kwargs`: собирает лишние именованные аргументы в `dict`.

Ключевой аспект для интервью: умение проектировать **понятные и устойчивые API**.

Практические правила:
- Явно отделяйте обязательные параметры от опциональных.
- Для публичных функций часто полезно ограничивать часть параметров как keyword-only (через `*`).
- Избегайте «магии» в `**kwargs`, если это ухудшает читаемость и автодополнение.

In [ ]:
def train_model(X, y, epochs=10, lr=1e-3, *, verbose=False, **kwargs):
    optimizer = kwargs.get('optimizer', 'adam')
    weight_decay = kwargs.get('weight_decay', 0.0)
    print(f"epochs={epochs}, lr={lr}, verbose={verbose}")
    print(f"optimizer={optimizer}, weight_decay={weight_decay}")
    return {'status': 'ok', 'n_samples': len(X)}

X = [1, 2, 3]
y = [0, 1, 0]

result_1 = train_model(X, y)
result_2 = train_model(X, y, 20, 5e-4, verbose=True, optimizer='sgd')
print(result_1, result_2)


def metric_report(name, *values, normalize=False, **meta):
    scale = max(values) if normalize and values else 1
    normalized = [v / scale for v in values] if scale else list(values)
    return {
        'name': name,
        'values': values,
        'normalized': normalized,
        'meta': meta,
    }

print(metric_report('loss', 0.9, 0.6, 0.4, normalize=True, split='val'))

### Разбор
- `train_model` показывает смешанный стиль аргументов.
- Символ `*` в сигнатуре делает `verbose` **только именованным** аргументом — это снижает число ошибок при вызове.
- `**kwargs` удобно использовать для проброса параметров оптимизатора, но злоупотребление скрывает контракт функции.

### Производительность и память
- Создание `tuple` для `*args` и `dict` для `**kwargs` имеет накладные расходы.
- В высокочастотных вызовах лучше использовать фиксированную сигнатуру.

### Interview-вопросы
1. Чем positional-only и keyword-only параметры помогают стабильности API?
2. Когда `**kwargs` оправдан, а когда вреден?
3. Почему изменение порядка параметров в публичной функции — потенциально breaking change?

### Мини-задачи
1. Перепишите функцию инференса так, чтобы `batch_size` и `device` были keyword-only.
2. Добавьте в `metric_report` валидацию типов входов.

### Edge cases
- Дублирование аргумента: `f(1, x=1)` вызовет `TypeError`.
- Неожиданные ключи в `**kwargs` могут молча игнорироваться — источник скрытых багов.

## 2) Распаковка аргументов

### Теория
Распаковка позволяет передавать коллекции в функцию как отдельные аргументы:
- `*iterable` → позиционные аргументы
- `**mapping` → именованные аргументы

Полезно для композиции пайплайнов, фабрик объектов, конфигурационных словарей.

In [ ]:
def build_experiment(model_name, dataset, epochs, lr, seed=42):
    return {'model': model_name, 'dataset': dataset, 'epochs': epochs, 'lr': lr, 'seed': seed}

positional = ('resnet18', 'cifar10', 30, 1e-3)
keyword = {'seed': 7}

exp = build_experiment(*positional, **keyword)
print(exp)

a, *middle, b = [1, 2, 3, 4, 5]
print(a, middle, b)

### Разбор
- `*positional` раскрывает `tuple` по позициям.
- `**keyword` раскрывает словарь как именованные аргументы.
- Важно отслеживать конфликты ключей.

### Производительность и память
- Распаковка создаёт временные объекты при мердже структур.
- Для больших конфигов избегайте лишнего копирования в горячем коде.

### Interview-вопросы
1. Что произойдёт при конфликте аргументов в `f(*args, **kwargs)`?
2. Чем отличается распаковка в вызове функции и в присваивании?

### Мини-задачи
1. Реализуйте безопасный merge конфигов с проверкой пересечений ключей.
2. Напишите helper `call_with_logging(func, *args, **kwargs)`.

### Edge cases
- `TypeError: got multiple values for argument`.
- `**kwargs` требует mapping со строковыми ключами.

## 3) Изменяемые значения по умолчанию: проблема и решение

### Теория
Значения параметров по умолчанию вычисляются **один раз** в момент определения функции.
Если default — изменяемый объект (`list`, `dict`, `set`), состояние разделяется между вызовами.

In [ ]:
def append_bad(x, bucket=[]):
    bucket.append(x)
    return bucket

print(append_bad(1))
print(append_bad(2))


def append_good(x, bucket=None):
    if bucket is None:
        bucket = []
    bucket.append(x)
    return bucket

print(append_good(1))
print(append_good(2))

### Разбор
- `append_bad` демонстрирует разделяемое состояние.
- Паттерн `None`-sentinel делает поведение предсказуемым.

### Производительность и память
- Осознанный shared-default можно использовать как кэш, но это рискованно для потоков и тестируемости.

### Interview-вопросы
1. Почему default-аргументы вычисляются один раз?
2. Как сделать кэш безопаснее (например, `lru_cache`)?

### Мини-задачи
1. Исправьте mutable-default баг в функции препроцессинга.
2. Реализуйте accumulator с ограничением размера.

### Edge cases
- Если `None` — валидное значение домена, используйте отдельный sentinel-объект.

## 4) Замыкания (closures) и внутреннее состояние

### Теория
Замыкание — функция, которая захватывает переменные из внешней области видимости.
Это способ хранить состояние без класса.

In [ ]:
def make_threshold_filter(threshold):
    calls = 0

    def _filter(values):
        nonlocal calls
        calls += 1
        selected = [v for v in values if v >= threshold]
        return selected, calls

    return _filter

f = make_threshold_filter(0.5)
print(f([0.1, 0.7, 0.8]))
print(f([0.6, 0.2]))


def make_multiplier(k):
    return lambda x: x * k

mul3 = make_multiplier(3)
print(mul3(10))

### Разбор
- `nonlocal` позволяет изменять захваченную переменную `calls`.
- Замыкания удобны для lightweight-stateful логики.

### Производительность и память
- Замыкание удерживает ссылки на захваченные объекты и продлевает их жизнь.
- Для сложного состояния класс обычно читабельнее.

### Interview-вопросы
1. Отличие `nonlocal` от `global`.
2. Когда closure лучше класса?

### Мини-задачи
1. Реализуйте closure-таймер со средней длительностью вызова.
2. Сделайте closure для clipping градиентов.

### Edge cases
- Late binding в циклах для lambda/inner functions.

## 5) Декораторы: с нуля, с аргументами и без

### Теория
Декоратор — функция, принимающая функцию и возвращающая новую функцию.
Типовые кейсы: логирование, ретраи, тайминг, кэширование.

In [ ]:
from functools import wraps
from time import perf_counter


def timing(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        t0 = perf_counter()
        out = func(*args, **kwargs)
        dt = perf_counter() - t0
        print(f"[timing] {func.__name__}: {dt:.6f}s")
        return out
    return wrapper


def repeat(n=2):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            result = None
            for _ in range(n):
                result = func(*args, **kwargs)
            return result
        return wrapper
    return decorator


@timing
@repeat(3)
def heavy_op(x):
    s = 0
    for i in range(100000):
        s += (i * x) % 7
    return s

print(heavy_op(5))
print(heavy_op.__name__)

### Разбор
- `timing` — декоратор без аргументов.
- `repeat(n)` — фабрика декораторов.
- `@wraps` сохраняет имя, docstring и другие метаданные функции.

### Производительность и память
- Каждый слой декоратора добавляет накладные расходы вызова.
- В inner-loop декораторы могут быть заметны.

### Interview-вопросы
1. Зачем нужен `functools.wraps`?
2. Как порядок декораторов влияет на результат?

### Мини-задачи
1. Напишите `retry(max_attempts, exceptions=...)`.
2. Добавьте декоратор аудита аргументов без утечки чувствительных данных.

### Edge cases
- Декорирование методов требует корректной обработки `self`/`cls`.
- Ошибки в декораторе могут скрыть root cause.

## 6) LEGB rule

### Теория
Python ищет имена в порядке: Local → Enclosing → Global → Builtins.
Непонимание LEGB часто приводит к `UnboundLocalError`.

In [ ]:
x = 'global_x'

def outer():
    x = 'enclosing_x'

    def inner():
        x = 'local_x'
        return x

    return inner(), x

print(outer())
print(x)

counter = 0

def good_increment_global():
    global counter
    counter += 1

good_increment_global()
print(counter)

### Разбор
- Локальные имена затеняют внешние.
- Для записи в глобальную переменную нужен `global`.

### Производительность и память
- Локальные lookup обычно быстрее глобальных.

### Interview-вопросы
1. Почему чтение глобальной переменной возможно без `global`, а запись — нет?
2. Как LEGB связан с closures?

### Мини-задачи
1. Исправьте пример с `UnboundLocalError` через `global` и `nonlocal`.
2. Покажите риск затенения `list`/`dict`.

### Edge cases
- Переопределение builtins ломает ожидаемое поведение.

## 7) Ловушки областей видимости

### Теория
Частые ошибки: late binding в циклах, затенение имён, мутация внешнего состояния без контроля.

In [ ]:
funcs_bad = []
for i in range(3):
    funcs_bad.append(lambda x: x + i)
print([f(10) for f in funcs_bad])

funcs_good = []
for i in range(3):
    funcs_good.append(lambda x, i=i: x + i)
print([f(10) for f in funcs_good])

### Разбор
- В плохом примере все функции используют последнее значение `i`.
- Фиксация `i=i` захватывает текущее значение на шаге цикла.

### Производительность и память
- Массовое создание closures может быть затратным.

### Interview-вопросы
1. Что такое late binding?
2. Почему default-параметр решает проблему?

### Мини-задачи
1. Исправьте callback-генератор для пайплайна аугментаций.
2. Перепишите через `functools.partial`.

### Edge cases
- Аналогичные баги часто встречаются в async/callback коде.

## 8) `classmethod` vs `staticmethod` vs instance methods

### Теория
- Instance method работает с `self`.
- Class method работает с `cls` (например, alternate constructor).
- Static method — логически связанная утилита без доступа к состоянию класса/экземпляра.

In [ ]:
class FeatureScaler:
    default_eps = 1e-8

    def __init__(self, mean, std):
        self.mean = mean
        self.std = std

    def transform(self, x):
        return (x - self.mean) / (self.std + self.default_eps)

    @classmethod
    def from_data(cls, data):
        mean = sum(data) / len(data)
        var = sum((v - mean) ** 2 for v in data) / len(data)
        std = var ** 0.5
        return cls(mean, std)

    @staticmethod
    def is_valid_number(x):
        return isinstance(x, (int, float))

scaler = FeatureScaler.from_data([1, 2, 3, 4, 5])
print(scaler.transform(3))
print(FeatureScaler.is_valid_number(3.14))

### Разбор
- `from_data` — типичный classmethod для альтернативного конструктора.
- `is_valid_number` — утилита домена.

### Производительность и память
- Разница в скорости между типами методов обычно вторична относительно дизайна API.

### Interview-вопросы
1. Когда classmethod лучше фабрики вне класса?
2. Когда staticmethod лучше вынести в модуль?

### Мини-задачи
1. Добавьте `from_dict` как classmethod.
2. Сравните читаемость при переносе staticmethod в утилитный модуль.

### Edge cases
- Переопределение classmethod в наследниках даёт полиморфное конструирование.

## 9) MRO и множественное наследование

### Теория
MRO (Method Resolution Order) задаёт порядок поиска методов.
Python использует C3 linearization.

In [ ]:
class LoggerMixin:
    def process(self, x):
        print('LoggerMixin')
        return super().process(x)

class ValidateMixin:
    def process(self, x):
        if x < 0:
            raise ValueError('x must be >= 0')
        print('ValidateMixin')
        return super().process(x)

class BaseProcessor:
    def process(self, x):
        print('BaseProcessor')
        return x * 2

class MyProcessor(LoggerMixin, ValidateMixin, BaseProcessor):
    pass

p = MyProcessor()
print(MyProcessor.__mro__)
print(p.process(5))

### Разбор
- Кооперативный `super()` позволяет chain-of-responsibility через mixins.
- Порядок базовых классов критичен.

### Производительность и память
- MRO lookup быстрый, но сложные иерархии тяжело поддерживать.

### Interview-вопросы
1. Что сломается без `super()` в одном из mixin?
2. Как C3 снижает неоднозначность?

### Мини-задачи
1. Добавьте `MetricsMixin` и предскажите порядок вызовов.
2. Перепишите пример через композицию.

### Edge cases
- Diamond inheritance без кооперативного `super()` ведёт к пропуску логики.

## 10) Dunder-методы: `__str__`, `__repr__`, `__eq__`, `__hash__`, ...

### Теория
Dunder-методы определяют интеграцию объекта в протоколы Python: печать, сравнение, хэширование, сортировку.

In [ ]:
class ModelConfig:
    def __init__(self, name, lr):
        self.name = name
        self.lr = lr

    def __repr__(self):
        return f"ModelConfig(name={self.name!r}, lr={self.lr!r})"

    def __str__(self):
        return f"{self.name} (lr={self.lr})"

    def __eq__(self, other):
        if not isinstance(other, ModelConfig):
            return NotImplemented
        return (self.name, self.lr) == (other.name, other.lr)

    def __hash__(self):
        return hash((self.name, self.lr))

c1 = ModelConfig('xgboost', 0.1)
c2 = ModelConfig('xgboost', 0.1)
print(repr(c1))
print(str(c1))
print(c1 == c2)
print({c1, c2})

### Разбор
- `__repr__` — для разработчика, `__str__` — для пользователя.
- Контракт равенства и хэша обязателен для корректной работы set/dict.

### Производительность и память
- Сложный `__hash__` на больших структурах может быть дорогим.
- Hashable-объекты должны быть логически неизменяемыми.

### Interview-вопросы
1. Почему после переопределения `__eq__` объект может стать unhashable?
2. Когда возвращать `NotImplemented`?

### Мини-задачи
1. Добавьте `__lt__` для сортировки конфигов.
2. Реализуйте иммутабельный вариант класса.

### Edge cases
- `NaN` нарушает интуицию сравнения (`nan != nan`).

## 11) Dataclasses

### Теория
`dataclasses` убирают boilerplate в классах-данных и поддерживают декларативный стиль описания сущностей.

In [ ]:
from dataclasses import dataclass, field
from typing import List

@dataclass(frozen=True)
class ExperimentConfig:
    model: str
    lr: float
    features: List[str] = field(default_factory=list)

cfg = ExperimentConfig(model='bert', lr=2e-5, features=['title', 'body'])
print(cfg)

### Разбор
- `frozen=True` добавляет защиту от случайной модификации.
- `default_factory=list` корректно создаёт новый список на каждый экземпляр.

### Производительность и память
- `slots=True` может снизить потребление памяти для большого числа объектов.

### Interview-вопросы
1. Чем dataclass отличается от `NamedTuple` и pydantic?
2. Когда frozen-конфиги полезны в ML pipelines?

### Мини-задачи
1. Добавьте `__post_init__` с валидацией диапазона `lr`.
2. Создайте dataclass для параметров feature engineering.

### Edge cases
- Frozen dataclass не защищает от мутации вложенного списка.

## 12) Процесс создания объекта: `__new__` и `__init__`

### Теория
`__new__` создаёт объект, `__init__` его инициализирует.
`__new__` обычно нужен для immutable типов, singleton/flyweight и контроля аллокаций.

In [ ]:
class Singleton:
    _instance = None

    def __new__(cls, *args, **kwargs):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
        return cls._instance

    def __init__(self, value):
        self.value = value

a = Singleton(10)
b = Singleton(99)
print(a is b)
print(a.value, b.value)

### Разбор
- `__new__` возвращает один и тот же экземпляр.
- `__init__` вызывается при каждом вызове конструктора и может перезаписать состояние.

### Производительность и память
- Пулинг объектов может снизить аллокации, но усложняет потокобезопасность.

### Interview-вопросы
1. Почему singleton часто считают антипаттерном?
2. Как сделать одноразовую инициализацию?

### Мини-задачи
1. Добавьте флаг инициализации, чтобы `__init__` выполнялся один раз.
2. Реализуйте flyweight для повторяющихся токенов.

### Edge cases
- Наследование singleton-классов может привести к неожиданной семантике экземпляров.

## 13) SOLID в Python с практическими примерами

### Теория
SOLID помогает проектировать расширяемые и тестируемые системы.

- **S**: одна ответственность.
- **O**: открыт для расширения, закрыт для изменения.
- **L**: корректная подстановка подтипов.
- **I**: узкие интерфейсы вместо монолитных.
- **D**: зависимость от абстракций.

In [ ]:
from abc import ABC, abstractmethod

class Model(ABC):
    @abstractmethod
    def predict(self, x):
        ...

class LinearModel(Model):
    def predict(self, x):
        return 0.5 * x + 1

class TreeModel(Model):
    def predict(self, x):
        return 1 if x > 0.7 else 0

class InferenceService:
    def __init__(self, model: Model):
        self.model = model

    def run(self, batch):
        return [self.model.predict(x) for x in batch]

print(InferenceService(LinearModel()).run([0.1, 1.0]))
print(InferenceService(TreeModel()).run([0.1, 1.0]))

### Разбор
- DIP: `InferenceService` зависит от абстракции `Model`, а не от конкретной реализации.
- OCP: можно добавить новую модель без изменения сервиса.
- SRP: сервис отвечает за orchestration инференса, а не за математику модели.

### Производительность и память
- Абстракции добавляют косвенность, но дают масштабируемость архитектуры.

### Interview-вопросы
1. Пример нарушения LSP в ML inference API.
2. Как применить ISP к online/batch inference?
3. Когда SOLID избыточен?

### Мини-задачи
1. Добавьте отдельный интерфейс `BatchModel` (ISP).
2. Вынесите постпроцессинг в стратегию (OCP).

### Edge cases
- Overengineering: слишком много абстракций в маленьком скрипте ухудшают скорость разработки.

## Итоговый чек-лист перед собеседованием

- Объясняю сигнатуры и компромиссы `*args/**kwargs`.
- Показываю mutable-default bug и исправление.
- Уверенно разбираю closures, decorators, LEGB и scope pitfalls.
- Отличаю instance/class/static methods и MRO в multiple inheritance.
- Понимаю dunder-контракты, dataclasses и процесс создания объекта.
- Применяю SOLID прагматично для production ML-сервисов.